In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "muhlenbeck2015gaze")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "muhlenbeck_full_raw.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)
    

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)


In [3]:
# df['Participant'].unique()

In [4]:


df['study_id']="muhlenbeck2015gaze"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)



In [5]:
df.columns =df.columns.str.replace(' ', '_')
# df.columns

In [6]:
df['recording_date'].unique

df[['day','month', 'year']] = df['recording_date'].str.split('.',expand=True)

In [7]:
df['file_name'].unique()
df[['condition', 'file1','file2']] = df['file_name'].str.split('\\',expand=True)

In [8]:
pathway_gen = os.path.abspath(data["python_files"])
comp_path_name_errors = os.path.join(pathway_gen, "muhlenbeck_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['participant'] = df['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='participant', right_on='name', how='left')

# df['participant'].unique()


In [9]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 

df= df.merge(ape_dob,left_on='participant', right_on='name', how='left') #insert dob of participants
df['dodc'] = df['year'].astype(str) + '-' + df['month'].astype(str) + '-' + df['day'].astype(str)
df['dodc'] = pd.to_datetime(df['dodc'])
df['dob'] = pd.to_datetime(df['dob'])
df['age_in_years'] = (df['dodc'] - df['dob']).dt.days//365
df['participant'].unique()

human_participant_drop = [ 'johanna_vp1', 'test', 'emma_vp5', 'rosha_vp6', 'hh_vp4',
       'test_linda', 'victor_vp3', 'kai_vp4', 'hh_vp5', 'hh_vp1',
       'hh_vp2', 'hh_vp6', 'hh_vp3', 'hh_vp7']

remdf=[ 'johanna_vp1', 'test', 'emma_vp5', 'rosha_vp6', 'hh_vp4',
       'test_linda', 'victor_vp3', 'kai_vp4', 'hh_vp5', 'hh_vp1',
       'hh_vp2', 'hh_vp6', 'hh_vp3', 'hh_vp7']
df = df[~df.participant.isin(remdf)]
df['participant'].unique()

array(['bimbo', 'padana', 'dokana', 'tanah', 'raja', 'pini', 'suaq',
       'batak'], dtype=object)

In [10]:
df['condition'].unique()

array(['farben_musik_u1', 'farben_musik_u2', 'farben_musik_u3',
       'farben_musik_u4', 'nur_farben_bilder_u1', 'nur_farben_bilder_u2',
       'nur_musik_u1', 'nur_musik_u2', 'nur_musik_u3', 'nur_musik_u4'],
      dtype=object)

In [11]:


studyID_standardized=df[['study_id', 'year','month','day',
       'participant','age_in_years','sex','species','condition',

       'recording_name', 'recording_date', 
       'recording_resolution', 'export_date', 
       # 'filter_settings',
         'eye', 'validity', 'fixation_filter',
       'velocity_threshold', 'distance_threshold',
       
       'timestamp', 'datetimestamp', 'datetimestampstartoffset', 'number',
       'gazepointxleft', 'gazepointyleft', 'camxleft', 'camyleft',
       'distanceleft', 'pupilleft', 'validityleft', 'gazepointxright',
       'gazepointyright', 'camxright', 'camyright', 'distanceright',
       'pupilright', 'validityright', 'fixationindex', 'gazepointx',
       'gazepointy', 
       # 'event', 'eventkey', 'data1', 'data2', 'descriptor',
       'stimuliname',
       #   'stimuliid',
           'mediawidth', 'mediaheight', 'mediaposx',
       'mediaposy', 'mappedfixationpointx', 'mappedfixationpointy',
       'fixationduration', 
       # 'aoiids', 'aoinames', 'webgroupimage',
       'mappedgazedatapointx', 'mappedgazedatapointy', 'microsecondtimestamp',
       'absolutemicrosecondtimestamp' ]]



comp_out_path_stand = os.path.join(out_pathway, 'muhlenbeck2015gaze_standardized.csv')
studyID_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =studyID_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
studyID_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'muhlenbeck2015gaze_glossary.csv')
studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
